# Data Inspection

In [ ]:
%store -r items_tagged_new
top_80_percent = items_tagged_new

In [ ]:
pba_substring_set = ['Beyond', 'Impossible', 'Vegan', 'Veggie', 'Jackfruit']
def filter_substrings(text):
    matches = [substr for substr in pba_substring_set if substr in text]
    return ' '.join(matches)
pba_substrings = before_after_details['first_plant_based_mention'].apply(filter_substrings).rename('substring')
before_after_details_substrings = pd.concat([before_after_details, pba_substrings], axis=1)

In [ ]:
item_in_top_total = 0
top_items_from_sales = []
top_item_percentages = []

for loc_id in tqdm(restaurants_by_4m_coverage):

    # Setup
    top_items = set(top_80_percent.query('location_id in @loc_id')['item_name'].tolist())
    df = sales_and_menu_data[loc_id]
    promo_word = before_after_details.loc[loc_id,'first_plant_based_mention']
    
    # Naive
    item_in_top_total += int(promo_word in top_items)
    
    ######
    top_item_sales = df[df['item_name'].isin(top_items)]
    
    top_modifications = top_item_sales['item_modifications']
    top_items = top_item_sales['item_name']
    
    promo_substring = before_after_details_substrings.loc[loc_id, 'substring']
    
    df_containing_promo = df[df['item_name'].str.contains(promo_substring) | df['item_modifications'].str.contains(promo_substring)]
    
    potential_pbas = df_containing_promo['item_name'].drop_duplicates()
    
    top_80_item_percent = potential_pbas.isin(top_items).sum() / potential_pbas.size
    
    top_80_modification_percent = potential_pbas.isin(top_modifications).sum() / potential_pbas.size
    
    top_item_percentages.append((loc_id, top_80_item_percent, top_80_modification_percent))
    
    #top_items_from_sales.append(top_item_sales)
    #top_items_from_sales += df[df['item_name'].isin(top_items)]['item_name'].unique().tolist()
    
pd.DataFrame(top_item_percentages, columns=['location_id', 'pct_in_top_items', 'pct_in_top_modifications'])

In [ ]:
k=22
for loc_id in restaurants_by_4m_coverage[k:k+1]:
    
    promo = before_after_details.loc[loc_id, 'first_plant_based_mention']
    promo_substring = before_after_details_substrings.loc[loc_id, 'substring']
    
    df = sales_and_menu_data[loc_id]
    df_containing_promo = df[df['item_name'].str.contains(promo_substring) | df['item_modifications'].str.contains(promo_substring)]
    #potential_promo_sales = df_containing_promo.drop_duplicates(subset=['item_name','item_modifications'])
    potential_promos = df_containing_promo.value_counts(subset=['item_name','item_modifications'])
    
    print(loc_id, promo)
    print(potential_promos.to_string())
    print("\n\n\n\n")

In [ ]:
promo_datetime = before_after_details_true.loc['S8MT0YGD2KTN9','cross_over_date']
promo_list = before_after_details_true.loc['S8MT0YGD2KTN9','promo_name']
promo_item_containing = sales_and_menu_data['S8MT0YGD2KTN9'][sales_and_menu_data['S8MT0YGD2KTN9']['item_name'].str.contains(promo_list) | sales_and_menu_data['S8MT0YGD2KTN9']['item_modifications'].str.title().str.contains(promo_list)]
plt.plot(promo_item_containing.resample('W')['item_quantity'].sum())
plt.axvline(x=promo_datetime, color='red', linestyle='--', label='Promo Date')
plt.title("Plant-Based Analog")
plt.xticks(rotation=70)
plt.show()

In [ ]:
vurger_conditions = ['item_modifications.str.title().str.contains("Vegan")', 
                     'item_modifications.str.title().str.contains("Vurger")',
                     'item_modifications.str.title().str.contains("V-Urger")',
                     'item_name.str.title().str.contains("Vurger")',
                     'item_name.str.title().str.contains("V-Urger")']
print(sales_and_menu_data['S8MT0YGD2KTN9'].query(' or '.join(vurger_conditions))[['item_name','item_modifications']].to_string())